In [1]:
%cd practicas/

/workspace/practicas


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


# Fase 1: Preparación de los datos

In [2]:
from pyspark.sql import SparkSession

spark = ( SparkSession.builder
         .appName("pr507")
         .master("spark://spark-master:7077")
         .getOrCreate()
         )
 
sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 11:26:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql.types import StructType, StructField, BooleanType, IntegerType, StringType, DoubleType, LongType, TimestampType, FloatType
from pyspark.sql import functions as f
from pyspark.sql import Window


schema_world = StructType([
    StructField("user_id", FloatType(), True),
    StructField("venue_id", FloatType(), True),
    StructField("rating", FloatType(), True),
    ])

df = (spark.read
             .format("csv")
             .schema(schema_world)
             .option("header", "True")
             .load("./data/ratings.csv"))
df.show(5)

train, test = (df.randomSplit([0.8, 0.2], seed=42))

+-------+--------+------+
|user_id|venue_id|rating|
+-------+--------+------+
|    1.0|     1.0|   5.0|
|    1.0|    51.0|   4.0|
|    1.0|    51.0|   2.0|
|    1.0|    51.0|   5.0|
|    1.0|    52.0|   5.0|
+-------+--------+------+
only showing top 5 rows



# Fase 2: Contruccion y busqueda del modelo optimo

In [4]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

als = ALS(
    userCol="user_id",
    itemCol="venue_id",
    ratingCol="rating",
    coldStartStrategy="drop"
)

grid_params = ( ParamGridBuilder()
    .addGrid(als.rank, [5, 10, 20])
    .addGrid(als.regParam, [0.01, 0.1])
    .addGrid(als.maxIter, [10])
    .build()
)

evaluador = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

validador_cruzado = CrossValidator(
    estimator=als,
    estimatorParamMaps=grid_params,
    evaluator=evaluador,
    numFolds=3
)

modelo = validador_cruzado.fit(train)

26/03/18 11:26:26 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


# Fase 3: Evlución de resultados

In [11]:
import numpy as np

combinaciones = modelo.getEstimatorParamMaps()

notas_rmse = modelo.avgMetrics

ranking=sorted(zip(notas_rmse, combinaciones), key=lambda x: x[0])
notaMin, mejoresParametros = ranking[0]

print("--- MEJOR DE TODAS LAS PRUEBAS ---")
for nota, parametros in ranking:
    if nota < notaMin:
        notaMin = nota
        mejoresParametros = parametros

print (f"\nRMSE Obtenido: {notaMin} ")
for param, valor in mejoresParametros.items():
    print (f" {param.name}: {valor}")

--- MEJOR DE TODAS LAS PRUEBAS ---

RMSE Obtenido: 1.8651719331297556 
 rank: 20
 regParam: 0.1
 maxIter: 10


In [17]:
from pyspark.sql.functions import explode, col

users_df = spark.createDataFrame([(1,)], ["user_id"])

user_recs = modelo.bestModel.recommendForUserSubset(users_df, 15)

recommendations = user_recs.withColumn("rec", explode("recommendations")).select(
        col("user_id").alias("Usuario"),
        col("venue_id").alias("Restaurante"), # O el nombre del restaurante si hiciste el Join
        col("rating").alias("Nota Predicha")
    )

# 4. Mostramos el resultado en formato tabla
print(f"--- Top 15 Recomendaciones para el Usuario {1} ---")
recommendations.show(truncate=False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `venue_id` cannot be resolved. Did you mean one of the following? [`user_id`, `rec`, `recommendations`].;
'Project [user_id#5645 AS Usuario#5657, 'venue_id AS Restaurante#5658, 'rating AS Nota Predicha#5659]
+- Project [user_id#5645, recommendations#5649, rec#5653]
   +- Generate explode(recommendations#5649), false, [rec#5653]
      +- Project [user_id#5645, cast(recommendations#5646 as array<struct<venue_id:int,rating:float>>) AS recommendations#5649]
         +- Project [_1#5640 AS user_id#5645, _2#5641 AS recommendations#5646]
            +- SerializeFromObject [knownnotnull(assertnotnull(input[0, scala.Tuple2, true]))._1 AS _1#5640, mapobjects(lambdavariable(MapObject, ObjectType(class java.lang.Object), true, 400), if (isnull(validateexternaltype(lambdavariable(MapObject, ObjectType(class java.lang.Object), true, 400), StructField(_1,IntegerType,false), StructField(_2,FloatType,false),

In [ ]:
spark.stop()